# Diagnóstico de costo: MAXIM S-2 baseline vs shared residual

Este notebook **no entrena ni modifica checkpoints persistentes**. Restaura de
forma estricta los checkpoints fijados, construye ejecutables de benchmark y
caracteriza:

- parámetros totales, entrenables, ejecutados por la implementación actual y
  seleccionados semánticamente por tarea;
- FLOPs/bytes estimados por el compilador cuando el backend los expone;
- latencia sincronizada de inferencia (mediana y p95);
- memoria estática del ejecutable y memoria de proceso observada;
- costo de un paso real de entrenamiento: forward, pérdida, backward y update;
- trazas separadas del cronometraje y HLO comprimido para auditoría.

Se comparan el API actual (todas las salidas), un forward equivalente que devuelve
sólo la salida final y, para shared, ejecutables con tarea dinámica y tarea
especializada. Las rutas especializadas se aceptan sólo si su salida coincide
numéricamente con la ruta dinámica.

**Responsabilidades:** Codex genera y valida estáticamente este notebook. El usuario
lo ejecuta en Colab con GPU y Drive. Los números producidos no deben interpretarse
automáticamente como una decisión de arquitectura.

Salidas persistentes:

- `benchmark_results.json` (reanudable);
- `diagnostico_costo_maxim.md`;
- trazas de profiling;
- HLO comprimido de cada ejecutable.


In [ ]:
# 1. Entorno reproducible del fork usado para restaurar los checkpoints
import os
import subprocess
import sys
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

REPO_URL = "https://github.com/Matiata/maxim.git"
REPO_COMMIT = "98984438176afb21694dd1ec29b02fd9b40426c1"
REPO_DIR = Path("/content/maxim_benchmark_src")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
        text=True,
    ).strip()
    if remote.rstrip("/") != REPO_URL:
        raise RuntimeError(f"Repositorio inesperado: {remote}")

subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_COMMIT],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
if actual_commit != REPO_COMMIT:
    raise RuntimeError(f"Commit inesperado: {actual_commit}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "jax[cuda12]==0.10.2",
        "flax==0.11.2",
        "optax",
        "ml-collections==0.1.1",
        "einops",
        "Pillow",
        "nvidia-ml-py",
    ],
    check=True,
)
sys.path.insert(0, str(REPO_DIR))
print(
    "Entorno instalado. Si JAX ya estaba importado, reiniciar el runtime "
    "y ejecutar desde el inicio."
)


In [ ]:
# 2. Montar Drive, importar dependencias y exigir GPU
from google.colab import drive

drive.mount("/content/gdrive")

import collections
import gc
import gzip
import hashlib
import importlib.metadata
import json
import os
import platform
import subprocess
import threading
import time
import traceback
from collections.abc import Mapping
from pathlib import Path
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import traverse_util
from flax.core import unfreeze
from flax.training import checkpoints, train_state

from maxim.models.maxim import Model
from maxim.models.moe import ResidualExpertHead, SharedResidualMaximMoE
from maxim.models.router import RouterModel

expected_versions = {"jax": "0.10.2", "flax": "0.11.2"}
actual_versions = {
    name: importlib.metadata.version(name) for name in expected_versions
}
if actual_versions != expected_versions:
    raise RuntimeError(
        f"Versiones inesperadas: {actual_versions}; reiniciar el runtime."
    )

DEVICES = jax.devices()
GPU_DEVICES = [device for device in DEVICES if device.platform == "gpu"]
if not GPU_DEVICES:
    raise RuntimeError("Seleccionar un runtime GPU en Colab.")
DEVICE = GPU_DEVICES[0]
print("Dispositivo de benchmark:", DEVICE)


In [ ]:
# 3. Configuración visible y fijada antes de medir
TASKS = ["deblur", "dehaze", "denoise", "derain", "enhance"]
CHECKPOINT_ROOT = Path("/content/gdrive/MyDrive/Facultad/tesis/ckpts")
OUTPUT_DIR = CHECKPOINT_ROOT / "benchmark_maxim_baseline_shared_20261013"
RESULT_PATH = OUTPUT_DIR / "benchmark_results.json"
REPORT_PATH = OUTPUT_DIR / "diagnostico_costo_maxim.md"
PROFILE_DIR = OUTPUT_DIR / "profiles"
HLO_DIR = OUTPUT_DIR / "hlo"

MODEL_CONFIG = {
    "variant": "S-2",
    "dropout_rate": 0.1,
    "num_outputs": 3,
    "use_bias": True,
    "num_supervision_scales": 3,
}
MODEL_SPECS = [
    {
        "name": "baseline",
        "kind": "baseline",
        "relative_run": "maxim_no_moe/baseline_S-2_from_newsplit_ckpt_extra10",
        "epoch": 10,
        "expected_step": 20000,
        "val_macro": 31.874760357289738,
    },
    {
        "name": "shared",
        "kind": "shared",
        "relative_run": "moe_shared_residual_from_newsplit_baseline_extra10",
        "epoch": 8,
        "expected_step": 16000,
        "val_macro": 32.06335479306456,
    },
]

INFERENCE_SHAPES = [(256, 256), (512, 512)]
INFERENCE_BATCH_SIZE = 1
TRAIN_SHAPE = (256, 256)
TRAIN_BATCH_SIZE = 2
TRAIN_LEARNING_RATE = 1e-5
TRAIN_WEIGHT_DECAY = 1e-4
AUXILIARY_LOSS_WEIGHT = 1.0
SEED = 42

WARMUP_EXECUTIONS = 10
TIMED_EXECUTIONS = 100
TIMING_BLOCKS = 3
NVML_SAMPLE_INTERVAL_SECONDS = 0.005

RUN_BENCHMARKS = True
# Opt-in: en Colab el profiler puede abortar el kernel por incompatibilidad
# nativa aun cuando el benchmark ordinario funciona correctamente.
RUN_PROFILING = False
SAVE_HLO = True
MEASURE_SPECIALIZED_SHARED = True
FORCE_REMEASURE = False
MODEL_FILTER = ["baseline", "shared"]

# Para memoria estricta: ejecutar un modelo por runtime limpio. El JSON
# reanudable combina ambas ejecuciones si el protocolo coincide.
assert set(MODEL_FILTER).issubset({spec["name"] for spec in MODEL_SPECS})
assert WARMUP_EXECUTIONS >= 10
assert TIMED_EXECUTIONS >= 100
assert TIMING_BLOCKS >= 3
print("Resultado:", RESULT_PATH)


In [ ]:
# 4. Utilidades: JSON atómico, sincronización, entorno y memoria NVML
def json_safe(value):
    if isinstance(value, Mapping):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (float, int, str, bool)) or value is None:
        return value
    return str(value)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(json_safe(payload), ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    os.replace(temporary, path)


def block_tree(tree):
    def block_leaf(value):
        return value.block_until_ready() if hasattr(value, "block_until_ready") else value

    return jax.tree_util.tree_map(block_leaf, tree)


def command_output(command):
    try:
        return subprocess.check_output(command, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as error:
        return f"unavailable: {type(error).__name__}: {error}"


def device_memory_stats():
    try:
        return json_safe(DEVICE.memory_stats() or {})
    except Exception as error:
        return {"available": False, "error": f"{type(error).__name__}: {error}"}


def environment_payload():
    package_names = [
        "jax", "jaxlib", "flax", "optax", "orbax-checkpoint",
        "numpy", "nvidia-ml-py",
    ]
    packages = {}
    for name in package_names:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(),
        "packages": packages,
        "jax_devices": [str(device) for device in DEVICES],
        "device_kind": getattr(DEVICE, "device_kind", None),
        "platform": DEVICE.platform,
        "nvidia_smi": command_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
             "--format=csv,noheader"]
        ),
        "repo_url": REPO_URL,
        "repo_commit": REPO_COMMIT,
        "xla_python_client_preallocate": os.environ.get(
            "XLA_PYTHON_CLIENT_PREALLOCATE"
        ),
        "xla_python_client_allocator": os.environ.get(
            "XLA_PYTHON_CLIENT_ALLOCATOR"
        ),
        "default_matmul_precision": str(
            getattr(jax.config, "jax_default_matmul_precision", None)
        ),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }


class NvmlProcessSampler:
    """Muestrea memoria residente del proceso; no equivale a buffers vivos."""

    def __init__(self, interval_seconds=0.005):
        self.interval_seconds = interval_seconds
        self.samples = []
        self.error = None
        self._stop = threading.Event()
        self._thread = None
        self._pynvml = None
        self._handle = None

    def _sample_once(self):
        used = None
        processes = self._pynvml.nvmlDeviceGetComputeRunningProcesses(
            self._handle
        )
        for process in processes:
            if int(process.pid) == os.getpid():
                used = int(process.usedGpuMemory)
                break
        self.samples.append(0 if used is None else used)

    def _loop(self):
        while not self._stop.is_set():
            try:
                self._sample_once()
            except Exception as error:
                self.error = f"{type(error).__name__}: {error}"
                break
            self._stop.wait(self.interval_seconds)

    def __enter__(self):
        try:
            import pynvml

            self._pynvml = pynvml
            pynvml.nvmlInit()
            device_index = int(getattr(DEVICE, "id", 0))
            self._handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
            self._sample_once()
            self._thread = threading.Thread(target=self._loop, daemon=True)
            self._thread.start()
        except Exception as error:
            self.error = f"{type(error).__name__}: {error}"
        return self

    def __exit__(self, exc_type, exc, tb):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2.0)
        if self._pynvml is not None:
            try:
                self._sample_once()
            except Exception:
                pass
            try:
                self._pynvml.nvmlShutdown()
            except Exception:
                pass

    def summary(self):
        if not self.samples:
            return {"available": False, "error": self.error}
        return {
            "available": True,
            "sample_interval_seconds": self.interval_seconds,
            "sample_count": len(self.samples),
            "first_bytes": int(self.samples[0]),
            "minimum_bytes": int(min(self.samples)),
            "peak_bytes": int(max(self.samples)),
            "peak_minus_first_bytes": int(max(self.samples) - self.samples[0]),
            "meaning": (
                "NVML process-resident memory; may include JAX allocator "
                "reservation and can miss short peaks."
            ),
            "error": self.error,
        }


In [ ]:
# 5. Protocolo inmutable y resultado reanudable
PROTOCOL = {
    "schema_version": 1,
    "repo_url": REPO_URL,
    "repo_commit": REPO_COMMIT,
    "model_config": MODEL_CONFIG,
    "model_specs": MODEL_SPECS,
    "tasks": TASKS,
    "inference_shapes": INFERENCE_SHAPES,
    "inference_batch_size": INFERENCE_BATCH_SIZE,
    "train_shape": TRAIN_SHAPE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "train_learning_rate": TRAIN_LEARNING_RATE,
    "train_weight_decay": TRAIN_WEIGHT_DECAY,
    "auxiliary_loss_weight": AUXILIARY_LOSS_WEIGHT,
    "warmup_executions": WARMUP_EXECUTIONS,
    "timed_executions": TIMED_EXECUTIONS,
    "timing_blocks": TIMING_BLOCKS,
    "preallocation": False,
    "inference_inputs": "synthetic float32 tensors already resident on GPU",
    "training_step": "forward + normalized deep-supervision L1 + backward + AdamW update",
    "shared_routing": "oracle; dynamic plus task-specialized final-only executables",
    "timing_excludes": [
        "compilation", "Drive I/O", "checkpoint restoration",
        "CPU/GPU transfer", "PSNR/SSIM", "profiling",
    ],
}
PROTOCOL_SHA256 = hashlib.sha256(
    json.dumps(PROTOCOL, sort_keys=True).encode("utf-8")
).hexdigest()

CURRENT_ENVIRONMENT = environment_payload()


def comparable_environment(payload):
    packages = payload.get("packages", {})
    return {
        "device_kind": payload.get("device_kind"),
        "platform": payload.get("platform"),
        "nvidia_smi": payload.get("nvidia_smi"),
        "packages": {
            name: packages.get(name)
            for name in ("jax", "jaxlib", "flax", "optax")
        },
        "preallocate": payload.get("xla_python_client_preallocate"),
        "allocator": payload.get("xla_python_client_allocator"),
        "matmul_precision": payload.get("default_matmul_precision"),
    }


if RESULT_PATH.exists():
    REPORT = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
    if REPORT.get("protocol_sha256") != PROTOCOL_SHA256:
        raise RuntimeError(
            "Existe un benchmark de otro protocolo. Conservarlo y cambiar OUTPUT_DIR."
        )
    if comparable_environment(REPORT.get("environment", {})) != (
        comparable_environment(CURRENT_ENVIRONMENT)
    ):
        raise RuntimeError(
            "La corrida reanudada usa otra GPU, driver o entorno. "
            "Usar el mismo runtime comparable o cambiar OUTPUT_DIR."
        )
else:
    REPORT = {
        "status": "in_progress",
        "protocol_sha256": PROTOCOL_SHA256,
        "protocol": PROTOCOL,
        "environment": CURRENT_ENVIRONMENT,
        "models": {},
        "measurements": {},
        "profiles": {},
        "limitations": [
            (
                "NVML informa memoria residente/reservada del proceso, no sólo "
                "buffers vivos. Para comparar picos estrictamente, ejecutar un "
                "modelo por runtime limpio mediante MODEL_FILTER."
            ),
            (
                "cost_analysis y memory_analysis dependen de la versión/backend; "
                "campos ausentes se registran como no disponibles."
            ),
            (
                "Las entradas sintéticas sirven para costo y tiempo, no para "
                "calidad. La calidad se toma únicamente de metadata ya existente."
            ),
        ],
    }
    atomic_json(RESULT_PATH, REPORT)

PROFILE_DISABLED_REASON = (
    "Profiling JAX desactivado: en Colab una incompatibilidad nativa del "
    "PLUGIN_Profiler_Api puede abortar el kernel. Latencia, FLOPs, HLO y "
    "memoria NVML se miden sin el profiler."
)
if not RUN_PROFILING:
    if PROFILE_DISABLED_REASON not in REPORT.setdefault("limitations", []):
        REPORT["limitations"].append(PROFILE_DISABLED_REASON)
    for model_name in MODEL_FILTER:
        for suffix in (
            "inference/256x256/final_dynamic",
            "training/256x256_batch2",
        ):
            REPORT.setdefault("profiles", {}).setdefault(
                f"{model_name}/{suffix}",
                {"status": "disabled", "reason": PROFILE_DISABLED_REASON},
            )

# Una ejecución que quedó en 'started' terminó por interrupción externa
# (por ejemplo, reinicio del kernel) antes de poder actualizar el JSON.
for previous_execution in REPORT.setdefault("executions", []):
    if previous_execution.get("status") == "started":
        previous_execution["status"] = "interrupted"

REPORT["executions"].append(
    {
        "execution_id": time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()),
        "model_filter": list(MODEL_FILTER),
        "environment": CURRENT_ENVIRONMENT,
        "status": "started",
    }
)
atomic_json(RESULT_PATH, REPORT)


def persist_report():
    REPORT["updated_utc"] = time.strftime(
        "%Y-%m-%dT%H:%M:%SZ", time.gmtime()
    )
    atomic_json(RESULT_PATH, REPORT)


def measurement_complete(key):
    return (
        not FORCE_REMEASURE
        and REPORT.get("measurements", {}).get(key, {}).get("status")
        == "complete"
    )


def store_measurement(key, payload):
    REPORT.setdefault("measurements", {})[key] = json_safe(payload)
    persist_report()


print("Protocolo:", PROTOCOL_SHA256)
print("Estado reanudado:", REPORT["status"])


In [ ]:
# 6. Construcción y restauración estricta, sin fallback aleatorio
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def make_model(kind):
    backbone = Model(**MODEL_CONFIG)
    if kind == "baseline":
        return backbone
    if kind != "shared":
        raise ValueError(f"Modelo desconocido: {kind}")
    return SharedResidualMaximMoE(
        backbone,
        RouterModel(),
        [ResidualExpertHead(num_hidden_layers=2, zero_init_output=True)
         for _ in TASKS],
        routing_mode="oracle",
        correction_scale=1.0,
    )


def flatten_tree(tree):
    return traverse_util.flatten_dict(unfreeze(tree))


def validate_tree(expected, actual, label):
    wanted = flatten_tree(expected)
    found = flatten_tree(actual)
    missing = sorted(set(wanted) - set(found))
    extra = sorted(set(found) - set(wanted))
    if missing or extra:
        raise ValueError(
            f"{label}: faltan {missing[:5]}, sobran {extra[:5]}"
        )
    for key in wanted:
        if wanted[key].shape != found[key].shape:
            raise ValueError(
                f"{label}: shape incompatible en {key}: "
                f"{wanted[key].shape} != {found[key].shape}"
            )
        if np.dtype(wanted[key].dtype) != np.dtype(found[key].dtype):
            raise ValueError(
                f"{label}: dtype incompatible en {key}: "
                f"{wanted[key].dtype} != {found[key].dtype}"
            )


def hash_variables(variables):
    digest = hashlib.sha256()
    for key, value in sorted(flatten_tree(variables).items()):
        array = np.ascontiguousarray(np.asarray(value))
        if not np.isfinite(array).all():
            raise FloatingPointError(f"Variable no finita: {key}")
        digest.update(json.dumps(key).encode("utf-8"))
        digest.update(str(array.dtype).encode("ascii"))
        digest.update(str(array.shape).encode("ascii"))
        digest.update(array.tobytes())
    return digest.hexdigest()


def load_model_strict(spec):
    run_dir = CHECKPOINT_ROOT / spec["relative_run"]
    best_dir = run_dir / "best_checkpoint"
    checkpoint_path = best_dir / f"checkpoint_{spec['epoch']}"
    selection_path = best_dir / "best_metric.json"
    run_metadata_path = run_dir / "run_metadata.json"
    for required_path in (checkpoint_path, selection_path, run_metadata_path):
        if not required_path.exists():
            raise FileNotFoundError(required_path)

    selection = read_json(selection_path)
    if selection.get("selection_metric") != "psnr_macro":
        raise ValueError(f"{spec['name']}: selección no macro.")
    if int(selection.get("epoch", -1)) != spec["epoch"]:
        raise ValueError(f"{spec['name']}: época seleccionada cambió.")
    if abs(float(selection["psnr_macro"]) - spec["val_macro"]) > 1e-6:
        raise ValueError(f"{spec['name']}: PSNR macro seleccionado cambió.")
    if selection.get("task_names") != TASKS:
        raise ValueError(f"{spec['name']}: orden de tareas incompatible.")
    if selection.get("task_count") != [868, 66, 463, 360, 546]:
        raise ValueError(f"{spec['name']}: conteos de validación incompatibles.")
    run_metadata = read_json(run_metadata_path)
    if "split_manifest" not in run_metadata:
        raise ValueError(f"{spec['name']}: falta split_manifest.")
    split_manifest_sha256 = hashlib.sha256(
        json.dumps(run_metadata["split_manifest"], sort_keys=True).encode("utf-8")
    ).hexdigest()

    model = make_model(spec["kind"])
    dummy = jnp.zeros(
        (1, INFERENCE_SHAPES[0][0], INFERENCE_SHAPES[0][1], 3),
        dtype=jnp.float32,
    )
    init_kwargs = (
        {"task_id": jnp.zeros((1,), dtype=jnp.int32)}
        if spec["kind"] == "shared" else {}
    )
    template = jax.eval_shape(
        lambda: model.init(
            jax.random.PRNGKey(SEED), dummy, train=False, **init_kwargs
        )
    )
    raw = checkpoints.restore_checkpoint(
        ckpt_dir=str(checkpoint_path), target=None
    )
    if not isinstance(raw, Mapping) or "params" not in raw or "step" not in raw:
        raise ValueError(
            f"{spec['name']}: no se restauró un TrainState completo."
        )
    restored_step = int(np.asarray(raw["step"]))
    if restored_step != spec["expected_step"]:
        raise ValueError(
            f"{spec['name']}: step {restored_step} != {spec['expected_step']}"
        )

    variables = {"params": raw["params"]}
    if raw.get("batch_stats") is not None:
        variables["batch_stats"] = raw["batch_stats"]
    validate_tree(template, variables, spec["name"])
    fingerprint = hash_variables(variables)
    variables = jax.tree_util.tree_map(jnp.asarray, variables)

    return {
        "name": spec["name"],
        "kind": spec["kind"],
        "model": model,
        "variables": variables,
        "fingerprint_sha256": fingerprint,
        "restored_step": restored_step,
        "checkpoint_path": str(checkpoint_path),
        "selection": selection,
        "run_metadata": run_metadata,
        "split_manifest_sha256": split_manifest_sha256,
    }


print("Restauración estricta preparada; ningún optimizador fue restaurado.")


In [ ]:
# 7. Conteo de parámetros y auditoría de ejecución de expertos
def leaf_parameter_count(value):
    return int(np.prod(value.shape, dtype=np.int64))


def parameter_inventory(handle):
    flat = flatten_tree(handle["variables"]["params"])
    components = collections.defaultdict(
        lambda: {"parameters": 0, "bytes": 0, "tensors": 0}
    )
    dtype_counts = collections.Counter()
    for key, value in flat.items():
        count = leaf_parameter_count(value)
        item_size = int(np.dtype(value.dtype).itemsize)
        component = str(key[0])
        components[component]["parameters"] += count
        components[component]["bytes"] += count * item_size
        components[component]["tensors"] += 1
        dtype_counts[str(value.dtype)] += count

    total = int(sum(item["parameters"] for item in components.values()))
    total_bytes = int(sum(item["bytes"] for item in components.values()))
    payload = {
        "total_parameters": total,
        "trainable_parameters": total,
        "total_parameter_bytes": total_bytes,
        "tensor_count": len(flat),
        "dtype_parameter_counts": dict(dtype_counts),
        "components": dict(components),
    }

    if handle["kind"] == "baseline":
        payload["executed_current_parameters"] = total
        payload["semantic_selected_parameters_by_task"] = {
            task: total for task in TASKS
        }
    else:
        maxim_count = components["maxim"]["parameters"]
        semantic = {}
        for index, task in enumerate(TASKS):
            expert_key = f"experts_{index}"
            if expert_key not in components:
                raise ValueError(f"Falta {expert_key} en shared.")
            semantic[task] = (
                maxim_count + components[expert_key]["parameters"]
            )
        payload["executed_current_parameters"] = total
        payload["semantic_selected_parameters_by_task"] = semantic
        payload["execution_note"] = (
            "La implementación actual construye y apila las cinco "
            "correcciones antes de multiplicar por gates."
        )
    return payload


def audit_shared_source():
    source_path = REPO_DIR / "maxim" / "models" / "moe.py"
    source = source_path.read_text(encoding="utf-8")
    required_fragments = [
        "corrections = jnp.stack",
        "[expert(feats) for expert in self.experts]",
        "mixed_correction = jnp.sum",
    ]
    missing = [fragment for fragment in required_fragments if fragment not in source]
    if missing:
        raise RuntimeError(f"Cambió el forward shared: faltan {missing}")
    return {
        "source_path": str(source_path),
        "source_sha256": hashlib.sha256(source.encode("utf-8")).hexdigest(),
        "all_experts_materialized_before_selection": True,
        "evidence_fragments": required_fragments,
    }


SHARED_SOURCE_AUDIT = audit_shared_source()
print("Auditoría estática shared:", SHARED_SOURCE_AUDIT)


In [ ]:
# 8. Forward de inferencia y paso real de entrenamiento
class BenchmarkState(train_state.TrainState):
    batch_stats: Any = None


def resize_target_to(prediction, target):
    if prediction.shape == target.shape:
        return target
    scale_h = target.shape[1] // prediction.shape[1]
    scale_w = target.shape[2] // prediction.shape[2]
    return target[:, ::scale_h, ::scale_w, :]


def compute_supervision_losses(
    predictions, targets, num_scales=3, auxiliary_weight=1.0
):
    if not isinstance(predictions, (list, tuple)) or not predictions:
        raise ValueError("Se esperaban predicciones multietapa.")
    final_prediction = predictions[-1][-1]
    final_target = resize_target_to(final_prediction, targets)
    final_loss = jnp.mean(jnp.abs(final_prediction - final_target))
    auxiliary_sum = jnp.zeros_like(final_loss)
    auxiliary_weight_sum = 0.0
    last_stage = len(predictions) - 1
    for stage_index, stage_predictions in enumerate(predictions):
        for scale_index, prediction in enumerate(stage_predictions):
            is_final = (
                stage_index == last_stage
                and scale_index == len(stage_predictions) - 1
            )
            if is_final:
                continue
            scale_weight = 0.5 ** (num_scales - scale_index - 1)
            target_at_scale = resize_target_to(prediction, targets)
            auxiliary_sum += scale_weight * jnp.mean(
                jnp.abs(prediction - target_at_scale)
            )
            auxiliary_weight_sum += scale_weight
    denominator = jnp.maximum(
        jnp.asarray(auxiliary_weight_sum, dtype=final_loss.dtype),
        jnp.asarray(1e-8, dtype=final_loss.dtype),
    )
    auxiliary_loss = auxiliary_sum / denominator
    total = final_loss + auxiliary_weight * auxiliary_loss
    return total, final_loss, auxiliary_loss, final_prediction


def final_from_full_output(kind, output):
    predictions = output[0] if kind == "shared" else output
    return predictions[-1][-1]


def make_inference_function(handle, output_mode, static_task=None):
    model = handle["model"]
    kind = handle["kind"]
    if output_mode not in ("full", "final"):
        raise ValueError(output_mode)

    def inference(variables, batch_input, task_id):
        with jax.named_scope(
            f"{kind}_{output_mode}_"
            + ("dynamic" if static_task is None else f"task_{static_task}")
        ):
            if kind == "shared":
                effective_task = (
                    task_id if static_task is None
                    else jnp.full(
                        (batch_input.shape[0],),
                        static_task,
                        dtype=jnp.int32,
                    )
                )
                output = model.apply(
                    variables, batch_input, train=False,
                    task_id=effective_task,
                )
            else:
                output = model.apply(variables, batch_input, train=False)
            if output_mode == "full":
                return output
            return final_from_full_output(kind, output)

    return jax.jit(inference)


def create_benchmark_state(handle):
    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(
            learning_rate=TRAIN_LEARNING_RATE,
            weight_decay=TRAIN_WEIGHT_DECAY,
        ),
    )
    return BenchmarkState.create(
        apply_fn=handle["model"].apply,
        params=handle["variables"]["params"],
        tx=tx,
        batch_stats=handle["variables"].get("batch_stats"),
    )


def make_train_step(handle):
    kind = handle["kind"]
    num_scales = MODEL_CONFIG["num_supervision_scales"]

    def train_step(state, batch_input, batch_target, task_id, rng):
        def loss_fn(params):
            variables = {"params": params}
            if state.batch_stats is not None:
                variables["batch_stats"] = state.batch_stats
            kwargs = {"task_id": task_id} if kind == "shared" else {}
            rngs = {"dropout": rng}
            if state.batch_stats is not None:
                output, updates = state.apply_fn(
                    variables, batch_input, train=True, rngs=rngs,
                    mutable=["batch_stats"], **kwargs,
                )
                new_batch_stats = updates["batch_stats"]
            else:
                output = state.apply_fn(
                    variables, batch_input, train=True, rngs=rngs, **kwargs
                )
                new_batch_stats = None
            predictions = output[0] if kind == "shared" else output
            loss, final_loss, auxiliary_loss, _ = (
                compute_supervision_losses(
                    predictions, batch_target, num_scales=num_scales,
                    auxiliary_weight=AUXILIARY_LOSS_WEIGHT,
                )
            )
            return loss, (final_loss, auxiliary_loss, new_batch_stats)

        (loss, (final_loss, auxiliary_loss, new_batch_stats)), grads = (
            jax.value_and_grad(loss_fn, has_aux=True)(state.params)
        )
        new_state = state.apply_gradients(grads=grads)
        if new_batch_stats is not None:
            new_state = new_state.replace(batch_stats=new_batch_stats)
        return new_state, {
            "loss": loss,
            "final_loss": final_loss,
            "auxiliary_loss": auxiliary_loss,
        }

    return jax.jit(train_step)


print("Funciones de inferencia y entrenamiento preparadas.")


In [ ]:
# 9. Pruebas sintéticas de funciones puras (sin checkpoints)
fake_target = jnp.linspace(
    0.0, 1.0, 2 * 16 * 16 * 3, dtype=jnp.float32
).reshape(2, 16, 16, 3)
fake_predictions = [
    [fake_target[:, ::4, ::4, :],
     fake_target[:, ::2, ::2, :], fake_target],
    [fake_target[:, ::4, ::4, :],
     fake_target[:, ::2, ::2, :], fake_target],
]
synthetic_losses = compute_supervision_losses(
    fake_predictions, fake_target, num_scales=3
)
np.testing.assert_allclose(
    np.asarray(synthetic_losses[:3]), 0.0, atol=1e-7
)
blocked = block_tree({"value": jnp.asarray([1.0], dtype=jnp.float32)})
np.testing.assert_array_equal(np.asarray(blocked["value"]), np.asarray([1.0]))

temporary_json = Path("/tmp/maxim_benchmark_atomic_json_test.json")
atomic_json(temporary_json, {"finite": np.float32(1.0), "path": Path("/tmp")})
roundtrip = json.loads(temporary_json.read_text(encoding="utf-8"))
assert roundtrip == {"finite": 1.0, "path": "/tmp"}
temporary_json.unlink()
print("Pruebas sintéticas OK; no se consultaron checkpoints.")


In [ ]:
# 10. Compilación, análisis de costo/memoria y cronometraje sincronizado
def normalize_cost_analysis(compiled):
    try:
        analysis = compiled.cost_analysis()
    except Exception as error:
        return {"available": False,
                "error": f"{type(error).__name__}: {error}"}
    if isinstance(analysis, (list, tuple)):
        analysis = analysis[0] if len(analysis) == 1 else {
            f"entry_{index}": item
            for index, item in enumerate(analysis)
        }
    return {"available": True, "values": json_safe(analysis)}


def normalize_memory_analysis(compiled):
    try:
        analysis = compiled.memory_analysis()
    except Exception as error:
        return {"available": False,
                "error": f"{type(error).__name__}: {error}"}
    if analysis is None:
        return {"available": False, "error": "backend returned None"}
    if hasattr(analysis, "_asdict"):
        values = analysis._asdict()
    else:
        fields = [
            "generated_code_size_in_bytes", "argument_size_in_bytes",
            "output_size_in_bytes", "alias_size_in_bytes",
            "temp_size_in_bytes", "host_generated_code_size_in_bytes",
            "host_argument_size_in_bytes", "host_output_size_in_bytes",
            "host_alias_size_in_bytes", "host_temp_size_in_bytes",
        ]
        values = {
            field: getattr(analysis, field)
            for field in fields if hasattr(analysis, field)
        }
    return {"available": True, "values": json_safe(values)}


def save_hlo_text(compiled, key):
    if not SAVE_HLO:
        return None
    HLO_DIR.mkdir(parents=True, exist_ok=True)
    destination = HLO_DIR / f"{key.replace('/', '__')}.txt.gz"
    try:
        text = compiled.as_text()
        with gzip.open(destination, "wt", encoding="utf-8") as handle:
            handle.write(text)
        return {
            "path": str(destination),
            "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
            "characters": len(text),
            "expert_name_occurrences": {
                f"experts_{index}": text.count(f"experts_{index}")
                for index in range(len(TASKS))
            },
        }
    except Exception as error:
        return {"path": None,
                "error": f"{type(error).__name__}: {error}"}


def compile_callable(jitted_function, args, key):
    started = time.perf_counter()
    lowered = jitted_function.lower(*args)
    compiled = lowered.compile()
    compilation_seconds = time.perf_counter() - started
    sample_output = block_tree(compiled(*args))
    return {
        "compiled": compiled,
        "sample_output": sample_output,
        "compilation_seconds": compilation_seconds,
        "cost_analysis": normalize_cost_analysis(compiled),
        "memory_analysis": normalize_memory_analysis(compiled),
        "hlo": save_hlo_text(compiled, key),
    }


def benchmark_compiled(compiled, args, batch_size, metric_kind):
    for _ in range(WARMUP_EXECUTIONS):
        block_tree(compiled(*args))

    device_before = device_memory_stats()
    block_payloads = []
    all_milliseconds = []
    with NvmlProcessSampler(NVML_SAMPLE_INTERVAL_SECONDS) as sampler:
        for block_index in range(TIMING_BLOCKS):
            timings = []
            for _ in range(TIMED_EXECUTIONS):
                started = time.perf_counter_ns()
                output = compiled(*args)
                block_tree(output)
                timings.append(
                    (time.perf_counter_ns() - started) / 1e6
                )
            all_milliseconds.extend(timings)
            block_payloads.append(
                {
                    "block": block_index + 1,
                    "executions": len(timings),
                    "median_ms": float(np.median(timings)),
                    "p95_ms": float(np.percentile(timings, 95)),
                    "mean_ms": float(np.mean(timings)),
                }
            )
    payload = {
        "metric_kind": metric_kind,
        "warmup_executions": WARMUP_EXECUTIONS,
        "timed_executions_per_block": TIMED_EXECUTIONS,
        "blocks": block_payloads,
        "median_ms": float(np.median(all_milliseconds)),
        "p95_ms": float(np.percentile(all_milliseconds, 95)),
        "mean_ms": float(np.mean(all_milliseconds)),
        "batch_size": int(batch_size),
        "median_ms_per_image": float(
            np.median(all_milliseconds) / batch_size
        ),
        "p95_ms_per_image": float(
            np.percentile(all_milliseconds, 95) / batch_size
        ),
        "nvml_process_memory": sampler.summary(),
        "device_memory_stats_before": device_before,
        "device_memory_stats_after": device_memory_stats(),
    }
    if metric_kind == "training_step":
        payload["median_ms_per_step"] = payload["median_ms"]
        payload["p95_ms_per_step"] = payload["p95_ms"]
    return payload


def profile_compiled(compiled, args, key):
    if not RUN_PROFILING:
        return {"status": "disabled"}
    destination = PROFILE_DIR / (
        key.replace("/", "__") + "__" + str(time.time_ns())
    )
    try:
        destination.mkdir(parents=True, exist_ok=False)
        with jax.profiler.trace(
            str(destination), create_perfetto_link=False
        ):
            for _ in range(3):
                block_tree(compiled(*args))
        return {"status": "complete", "path": str(destination)}
    except Exception as error:
        return {
            "status": "unavailable", "path": str(destination),
            "error": f"{type(error).__name__}: {error}",
        }


print("Instrumentación preparada; aún no se ejecutó ningún benchmark.")


In [ ]:
# 11. Benchmark de inferencia: API actual, final-only y tarea especializada
def task_array(task_index, batch_size):
    return jnp.full((batch_size,), task_index, dtype=jnp.int32)


def compare_outputs(reference, candidate, label):
    reference = np.asarray(jax.device_get(reference))
    candidate = np.asarray(jax.device_get(candidate))
    if reference.shape != candidate.shape:
        raise ValueError(
            f"{label}: shapes {reference.shape} != {candidate.shape}"
        )
    maximum = float(np.max(np.abs(reference - candidate)))
    if maximum >= 1e-6:
        raise AssertionError(f"{label}: max_abs_diff={maximum:.3e}")
    return maximum


def run_inference_benchmarks(handle):
    kind = handle["kind"]
    variables = handle["variables"]
    rng = np.random.default_rng(SEED)

    for height, width in INFERENCE_SHAPES:
        shape_name = f"{height}x{width}"
        tasks_to_measure = TASKS if kind == "shared" else ["fixed"]
        expected_keys = [
            (
                f"{handle['name']}/inference/{shape_name}/"
                f"{output_mode}_dynamic/{task}"
            )
            for task in tasks_to_measure
            for output_mode in ("full", "final")
        ]
        if kind == "shared" and MEASURE_SPECIALIZED_SHARED:
            expected_keys.extend(
                f"{handle['name']}/inference/{shape_name}/final_static/{task}"
                for task in TASKS
            )
        profile_key = (
            f"{handle['name']}/inference/256x256/final_dynamic"
        )
        profile_pending = (
            RUN_PROFILING
            and (height, width) == (256, 256)
            and REPORT.get("profiles", {}).get(profile_key, {}).get("status")
            != "complete"
        )
        if all(measurement_complete(key) for key in expected_keys) and not (
            profile_pending
        ):
            print("Omitiendo resolución completa:", handle["name"], shape_name)
            continue
        batch_input = jax.device_put(
            rng.random(
                (INFERENCE_BATCH_SIZE, height, width, 3),
                dtype=np.float32,
            ), DEVICE
        )
        base_task = task_array(0, INFERENCE_BATCH_SIZE)

        compiled_modes = {}
        for output_mode in ("full", "final"):
            executable_key = (
                f"{handle['name']}/inference/{shape_name}/"
                f"{output_mode}_dynamic_executable"
            )
            function = make_inference_function(
                handle, output_mode, static_task=None
            )
            compiled_modes[output_mode] = compile_callable(
                function, (variables, batch_input, base_task),
                executable_key,
            )

        full_final = final_from_full_output(
            kind, compiled_modes["full"]["sample_output"]
        )
        final_only = compiled_modes["final"]["sample_output"]
        base_equivalence = compare_outputs(
            full_final, final_only,
            f"{handle['name']} {shape_name} full/final",
        )

        for task_index, task in enumerate(tasks_to_measure):
            effective_index = task_index if kind == "shared" else 0
            task_ids = task_array(effective_index, INFERENCE_BATCH_SIZE)
            for output_mode in ("full", "final"):
                key = (
                    f"{handle['name']}/inference/{shape_name}/"
                    f"{output_mode}_dynamic/{task}"
                )
                if measurement_complete(key):
                    print("Omitiendo medición completa:", key)
                    continue
                compiled_payload = compiled_modes[output_mode]
                store_measurement(key, {"status": "running"})
                try:
                    timing = benchmark_compiled(
                        compiled_payload["compiled"],
                        (variables, batch_input, task_ids),
                        INFERENCE_BATCH_SIZE, "inference",
                    )
                    store_measurement(
                        key,
                        {
                            "status": "complete",
                            "model": handle["name"],
                            "kind": kind,
                            "shape": [height, width],
                            "task": task,
                            "output_mode": output_mode,
                            "routing": (
                                "dynamic" if kind == "shared" else "fixed"
                            ),
                            "full_final_equivalence_max_abs_diff": base_equivalence,
                            "compilation_seconds": compiled_payload[
                                "compilation_seconds"
                            ],
                            "cost_analysis": compiled_payload["cost_analysis"],
                            "memory_analysis": compiled_payload["memory_analysis"],
                            "hlo": compiled_payload["hlo"],
                            "timing": timing,
                        },
                    )
                except Exception as error:
                    store_measurement(
                        key,
                        {
                            "status": "error",
                            "error": f"{type(error).__name__}: {error}",
                            "traceback": traceback.format_exc(),
                        },
                    )
                    raise

        if kind == "shared" and MEASURE_SPECIALIZED_SHARED:
            dynamic_compiled = compiled_modes["final"]["compiled"]
            for task_index, task in enumerate(TASKS):
                key = (
                    f"{handle['name']}/inference/{shape_name}/"
                    f"final_static/{task}"
                )
                if measurement_complete(key):
                    print("Omitiendo medición completa:", key)
                    continue
                static_function = make_inference_function(
                    handle, "final", static_task=task_index
                )
                static_payload = compile_callable(
                    static_function,
                    (variables, batch_input, base_task),
                    key + "_executable",
                )
                dynamic_task = task_array(
                    task_index, INFERENCE_BATCH_SIZE
                )
                dynamic_output = block_tree(
                    dynamic_compiled(
                        variables, batch_input, dynamic_task
                    )
                )
                equivalence = compare_outputs(
                    dynamic_output, static_payload["sample_output"],
                    f"shared {shape_name} dynamic/static {task}",
                )
                store_measurement(key, {"status": "running"})
                try:
                    timing = benchmark_compiled(
                        static_payload["compiled"],
                        (variables, batch_input, base_task),
                        INFERENCE_BATCH_SIZE, "inference",
                    )
                    store_measurement(
                        key,
                        {
                            "status": "complete",
                            "model": handle["name"],
                            "kind": kind,
                            "shape": [height, width],
                            "task": task,
                            "output_mode": "final",
                            "routing": "static_task_specialized",
                            "dynamic_static_equivalence_max_abs_diff": equivalence,
                            "compilation_seconds": static_payload[
                                "compilation_seconds"
                            ],
                            "cost_analysis": static_payload["cost_analysis"],
                            "memory_analysis": static_payload[
                                "memory_analysis"
                            ],
                            "hlo": static_payload["hlo"],
                            "timing": timing,
                        },
                    )
                except Exception as error:
                    store_measurement(
                        key,
                        {
                            "status": "error",
                            "error": f"{type(error).__name__}: {error}",
                            "traceback": traceback.format_exc(),
                        },
                    )
                    raise

        if RUN_PROFILING and (height, width) == (256, 256):
            profile_key = (
                f"{handle['name']}/inference/256x256/final_dynamic"
            )
            if REPORT.get("profiles", {}).get(profile_key, {}).get(
                "status"
            ) != "complete":
                REPORT.setdefault("profiles", {})[profile_key] = (
                    profile_compiled(
                        compiled_modes["final"]["compiled"],
                        (variables, batch_input, base_task),
                        profile_key,
                    )
                )
                persist_report()

        del compiled_modes, batch_input
        jax.clear_caches()
        gc.collect()


In [ ]:
# 12. Benchmark de un paso real de entrenamiento
def run_training_benchmark(handle):
    key = f"{handle['name']}/training/256x256_batch2/full_step"
    if measurement_complete(key):
        print("Omitiendo medición completa:", key)
        return

    rng_np = np.random.default_rng(SEED + 100)
    batch_input = jax.device_put(
        rng_np.random(
            (TRAIN_BATCH_SIZE, TRAIN_SHAPE[0], TRAIN_SHAPE[1], 3),
            dtype=np.float32,
        ), DEVICE
    )
    batch_target = jax.device_put(
        rng_np.random(
            (TRAIN_BATCH_SIZE, TRAIN_SHAPE[0], TRAIN_SHAPE[1], 3),
            dtype=np.float32,
        ), DEVICE
    )
    task_ids = jax.device_put(
        np.arange(TRAIN_BATCH_SIZE, dtype=np.int32) % len(TASKS),
        DEVICE,
    )
    rng = jax.random.PRNGKey(SEED + 200)
    state = create_benchmark_state(handle)
    step_function = make_train_step(handle)
    compiled_payload = compile_callable(
        step_function,
        (state, batch_input, batch_target, task_ids, rng),
        key + "_executable",
    )
    sample_state, sample_metrics = compiled_payload["sample_output"]
    if not np.isfinite(float(sample_metrics["loss"])):
        raise FloatingPointError("Pérdida no finita en preflight.")
    if int(sample_state.step) != int(state.step) + 1:
        raise AssertionError("El paso de optimizador no avanzó exactamente uno.")

    store_measurement(key, {"status": "running"})
    try:
        timing = benchmark_compiled(
            compiled_payload["compiled"],
            (state, batch_input, batch_target, task_ids, rng),
            TRAIN_BATCH_SIZE, "training_step",
        )
        store_measurement(
            key,
            {
                "status": "complete",
                "model": handle["name"],
                "kind": handle["kind"],
                "shape": list(TRAIN_SHAPE),
                "batch_size": TRAIN_BATCH_SIZE,
                "trainable_scope": "all checkpoint parameters",
                "optimizer": "clip_by_global_norm(1.0) + AdamW",
                "learning_rate": TRAIN_LEARNING_RATE,
                "weight_decay": TRAIN_WEIGHT_DECAY,
                "sample_loss": float(sample_metrics["loss"]),
                "compilation_seconds": compiled_payload[
                    "compilation_seconds"
                ],
                "cost_analysis": compiled_payload["cost_analysis"],
                "memory_analysis": compiled_payload["memory_analysis"],
                "hlo": compiled_payload["hlo"],
                "timing": timing,
            },
        )
    except Exception as error:
        store_measurement(
            key,
            {
                "status": "error",
                "error": f"{type(error).__name__}: {error}",
                "traceback": traceback.format_exc(),
            },
        )
        raise

    if RUN_PROFILING:
        profile_key = f"{handle['name']}/training/256x256_batch2"
        if REPORT.get("profiles", {}).get(profile_key, {}).get(
            "status"
        ) != "complete":
            REPORT.setdefault("profiles", {})[profile_key] = profile_compiled(
                compiled_payload["compiled"],
                (state, batch_input, batch_target, task_ids, rng),
                profile_key,
            )
            persist_report()

    del compiled_payload, step_function, state, sample_state
    del sample_metrics, batch_input, batch_target, task_ids
    jax.clear_caches()
    gc.collect()


In [ ]:
# 13. Orquestación: ejecutar sólo en Colab con checkpoints montados
if not RUN_BENCHMARKS:
    print("RUN_BENCHMARKS=False: no se restauran checkpoints ni se mide.")
else:
    specs_by_name = {spec["name"]: spec for spec in MODEL_SPECS}
    selected_specs = [specs_by_name[name] for name in MODEL_FILTER]
    for spec in selected_specs:
        print("\n" + "=" * 80)
        print("Restaurando y midiendo:", spec["name"])
        print("=" * 80)
        try:
            handle = load_model_strict(spec)
            known_split_hashes = {
                payload.get("split_manifest_sha256")
                for name, payload in REPORT.get("models", {}).items()
                if name != spec["name"] and payload.get("split_manifest_sha256")
            }
            if known_split_hashes and known_split_hashes != {
                handle["split_manifest_sha256"]
            }:
                raise ValueError("Baseline y shared no usan el mismo split manifest.")
            model_payload = {
                "status": "restored",
                "kind": handle["kind"],
                "checkpoint_path": handle["checkpoint_path"],
                "restored_step": handle["restored_step"],
                "variable_fingerprint_sha256": handle[
                    "fingerprint_sha256"
                ],
                "selection": handle["selection"],
                "split_manifest_sha256": handle["split_manifest_sha256"],
                "parameters": parameter_inventory(handle),
            }
            if handle["kind"] == "shared":
                model_payload["shared_source_audit"] = SHARED_SOURCE_AUDIT
            REPORT.setdefault("models", {})[spec["name"]] = model_payload
            persist_report()

            run_inference_benchmarks(handle)
            run_training_benchmark(handle)
            REPORT["models"][spec["name"]]["status"] = "complete"
            persist_report()
        except Exception as error:
            REPORT.setdefault("models", {}).setdefault(
                spec["name"], {}
            ).update(
                {
                    "status": "error",
                    "error": f"{type(error).__name__}: {error}",
                    "traceback": traceback.format_exc(),
                }
            )
            REPORT["executions"][-1]["status"] = "error"
            persist_report()
            raise
        finally:
            if "handle" in locals():
                del handle
            jax.clear_caches()
            gc.collect()

    completed_models = {
        name for name, payload in REPORT.get("models", {}).items()
        if payload.get("status") == "complete"
    }
    required_models = {spec["name"] for spec in MODEL_SPECS}
    REPORT["status"] = (
        "complete" if completed_models == required_models
        else "in_progress"
    )
    REPORT["completed_models"] = sorted(completed_models)
    REPORT["executions"][-1]["status"] = (
        "complete" if REPORT["status"] == "complete" else "partial"
    )
    persist_report()
    print("Estado final de esta ejecución:", REPORT["status"])


In [ ]:
# 14. Reporte legible derivado del JSON, sin tomar decisiones
def extract_flops(measurement):
    analysis = measurement.get("cost_analysis", {})
    if not analysis.get("available"):
        return None
    values = analysis.get("values", {})
    if isinstance(values, Mapping):
        for key in ("flops", "FLOPs", "flop_count"):
            if key in values:
                return values[key]
    return None


def extract_temp_bytes(measurement):
    analysis = measurement.get("memory_analysis", {})
    if not analysis.get("available"):
        return None
    return analysis.get("values", {}).get("temp_size_in_bytes")


def fmt(value, digits=3):
    if value is None:
        return "N/D"
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


report_lines = [
    "# Diagnóstico de costo MAXIM baseline/shared",
    "",
    f"- Estado: **{REPORT.get('status')}**",
    f"- Protocolo SHA-256: `{REPORT.get('protocol_sha256')}`",
    f"- Commit: `{REPO_COMMIT}`",
    (
        "- Nota: este reporte enumera mediciones; la selección de una "
        "sustitución se realiza después de revisar los artefactos."
    ),
    "",
    "## Parámetros",
    "",
    "| Modelo | Totales | Entrenables | Ejecutados actualmente |",
    "|---|---:|---:|---:|",
]
for name in ("baseline", "shared"):
    model_payload = REPORT.get("models", {}).get(name)
    if not model_payload:
        continue
    params = model_payload["parameters"]
    report_lines.append(
        f"| {name} | {params['total_parameters']} | "
        f"{params['trainable_parameters']} | "
        f"{params['executed_current_parameters']} |"
    )

report_lines.extend(
    [
        "", "## Mediciones", "",
        (
            "| Clave | Mediana (ms) | p95 (ms) | Mediana "
            "(ms/imagen) | FLOPs compilador | Temp. estática (bytes) |"
        ),
        "|---|---:|---:|---:|---:|---:|",
    ]
)
for key, measurement in sorted(REPORT.get("measurements", {}).items()):
    if measurement.get("status") != "complete":
        report_lines.append(
            f"| {key} ({measurement.get('status')}) | N/D | N/D | "
            "N/D | N/D | N/D |"
        )
        continue
    timing = measurement.get("timing", {})
    report_lines.append(
        f"| {key} | {fmt(timing.get('median_ms'))} | "
        f"{fmt(timing.get('p95_ms'))} | "
        f"{fmt(timing.get('median_ms_per_image'))} | "
        f"{fmt(extract_flops(measurement), 0)} | "
        f"{fmt(extract_temp_bytes(measurement), 0)} |"
    )

report_lines.extend(["", "## Limitaciones registradas", ""])
report_lines.extend(
    f"- {limitation}" for limitation in REPORT.get("limitations", [])
)
report_lines.extend(
    [
        "", "## Próximo paso", "",
        (
            "Entregar `benchmark_results.json`, este reporte y las trazas "
            "para revisar costos dominantes. No implementar S1/S2 antes de "
            "esa revisión."
        ),
        "",
    ]
)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")
print("JSON:", RESULT_PATH)
print("Reporte:", REPORT_PATH)
print("Perfiles:", PROFILE_DIR)
print("HLO:", HLO_DIR)


## Ejecución y entrega

1. Seleccionar una GPU L4 cuando esté disponible.
2. Ejecutar las celdas en orden. La compilación no forma parte del tiempo medido.
3. Si una ejecución se interrumpe, volver a ejecutar: las mediciones completas se
   omiten salvo que `FORCE_REMEASURE=True`.
4. Para memoria estricta, usar un runtime limpio con
   `MODEL_FILTER=["baseline"]`, reiniciar y repetir con
   `MODEL_FILTER=["shared"]`. El mismo JSON combina ambas corridas.
5. Entregar la carpeta
   `ckpts/benchmark_maxim_baseline_shared_20261013`, incluyendo JSON, Markdown,
   perfiles y HLO.

El notebook no guarda estados de entrenamiento ni altera los checkpoints fuente.
Las métricas de calidad no se recalculan aquí: se conserva la metadata de
validación ya fijada.
